In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(r"C:\Users\srush\OneDrive\Desktop\cripto.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\srush\\OneDrive\\Desktop\\cripto.csv'

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape


In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')


In [ ]:
df.head()

In [ ]:
df.columns = df.columns.str.lower()
df.head()


In [ ]:
df.isnull().sum()


In [ ]:
df = df[df['volume'] > 0]


In [ ]:
df.shape

In [ ]:
invalid_prices = df[
    (df['high'] < df[['open','close','low']].max(axis=1)) |
    (df['low'] > df[['open','close','high']].min(axis=1))
]

invalid_prices.shape

In [ ]:
df = df.drop(invalid_prices.index)


In [ ]:
df = df.drop_duplicates()


In [ ]:
q1 = df['volume'].quantile(0.25)
q3 = df['volume'].quantile(0.75)
iqr = q3 - q1

df = df[df['volume'] <= q3 + 1.5 * iqr]


In [ ]:
df = df.sort_values('timestamp')


In [ ]:
df.info()
df.describe()
df.head()


In [ ]:
df.to_csv("cleaned_crypto_data.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
df['close'].hist(bins=50)
plt.title("Distribution of Closing Price")
plt.xlabel("Close Price")
plt.ylabel("Frequency")
plt.show()


In [ ]:
df['month'] = df['timestamp'].dt.month

monthly_avg = df.groupby('month')['close'].mean()

plt.figure()
monthly_avg.plot(kind='bar')
plt.title("Average Closing Price by Month")
plt.xlabel("Month")
plt.ylabel("Avg Close Price")
plt.show()


In [ ]:
df['quarter'] = df['timestamp'].dt.quarter

quarter_volume = df.groupby('quarter')['volume'].sum()

plt.figure()
quarter_volume.plot(kind='pie', autopct='%1.1f%%')
plt.ylabel("")
plt.title("Trading Volume by Quarter")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['close'])
plt.title("Crypto Price Trend Over Time")
plt.xlabel("Time")
plt.ylabel("Close Price")
plt.show()

In [ ]:
df['ma_20'] = df['close'].rolling(20).mean()
df['ma_50'] = df['close'].rolling(50).mean()

plt.figure(figsize=(10,5))
plt.plot(df['timestamp'], df['close'], label='Close')
plt.plot(df['timestamp'], df['ma_20'], label='MA 20')
plt.plot(df['timestamp'], df['ma_50'], label='MA 50')
plt.legend()
plt.title("Moving Average Analysis")
plt.show()


In [ ]:
plt.figure()
plt.scatter(df['volume'], df['close'])
plt.xlabel("Volume")
plt.ylabel("Close Price")
plt.title("Price vs Volume Relationship")
plt.show()

In [ ]:
plt.figure()
plt.boxplot(df['volume'])
plt.title("Volume Outlier Detection")
plt.show()


In [ ]:
filtered_df = df[
    (df['timestamp'] >= '2022-01-01') &
    (df['timestamp'] <= '2022-12-31')
]


In [ ]:
high_volume = df[df['volume'] > df['volume'].quantile(0.90)]


In [ ]:
price_drop = df[df['close'] < df['open']]


In [ ]:
df['daily_return'] = df['close'].pct_change() * 100


In [ ]:
df[['daily_return']].describe()


In [ ]:
df[['open','high','low','close','volume']].corr()


In [ ]:
pip install psycopg2 sqlalchemy


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

In [ ]:
username = "postgres"
password = "....."
host = "localhost"
port = "5432"
database = "crypto_analysis"


In [ ]:
safe_password = urllib.parse.quote_plus(password)


In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{username}:{safe_password}@{host}:{port}/{database}"
)


In [ ]:
engine.connect()


In [ ]:
df.to_sql(
    name="crypto_prices",
    con=engine,
    if_exists="replace",
    index=False
)


In [ ]:
pd.read_sql("SELECT * FROM crypto_prices LIMIT 5", engine)


In [ ]:
query = "SELECT * FROM crypto_data LIMIT 5"
pd.read_sql(query, engine)



In [ ]:
pd.read_sql(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='public'",
    engine
)